# Experiment 042: GPT-2 124M technical pilot (Colab)

Pilot measures memory, timing, numerical movement for 30 GRPO updates. It performs no MMLU test evaluation and discards its weights.

## 0. Compute units and Drive

In [ ]:
NOTEBOOK_NAME = '042_pilot_colab_m124'
import glob, json, os, time

session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 326.36
unit_rate = 5.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])
    if not os.path.exists(open_path):
        record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                  'notebook': NOTEBOOK_NAME, 'colab_cu_before': cu_before,
                  'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(record, f, indent=2)
        print('saved open session:', session_id)


session 20260825_135245 | cu_before = 326.36
Mounted at /content/drive
closed previous session: 20260825_132546
saved open session: 20260825_135245


## 1. Runtime

In [ ]:
import shutil, subprocess, torch

gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(gpu.strip())
print(f'total vram: {total_gib:.1f} GiB')
assert 'A100' in gpu and total_gib >= 39, 'need an A100 40GB or larger runtime'
assert shutil.disk_usage('/content').free / 1024**3 >= 10, 'need at least 10 GiB free locally'
assert shutil.disk_usage(ROOT).free / 1024**3 >= 15, 'need at least 15 GiB free on Drive'


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-c24dab97-a381-5000-e6ea-32ba4aec7867)
total vram: 39.5 GiB


## 2. Repository and dependencies

In [ ]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


working in: /content/INM363-IndividualProject-Dissertation/001_Code


In [ ]:
import gc, os, subprocess, sys, torch

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
scratch_package = os.path.join(repo_root, '002_GPT2_From_Scratch')
if not os.path.isdir(scratch_package):
    scratch_package = os.path.join(repo_root, '003_My GPT From Scratch')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', scratch_package], check=True)
sys.path.insert(0, scratch_package)
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from gpt2_scratch.weights import REFERENCE_ALIAS, REFERENCE_REVISION
GPT2TokenizerFast.from_pretrained(REFERENCE_ALIAS, revision=REFERENCE_REVISION)
reference = GPT2LMHeadModel.from_pretrained(
    REFERENCE_ALIAS, revision=REFERENCE_REVISION, dtype=torch.float32)
del reference
gc.collect()
print('dependencies installed')


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

dependencies installed


## 3. Wiring checks

In [ ]:
import os, subprocess, sys

checks = subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    os.path.join(scratch_package, 'tests'),
    'tests/test_scratch_gpt_pipeline.py',
    'tests/test_042_m124.py',
    'tests/test_042_notebooks.py',
    '-k',
    'not test_042_notebooks_are_clean_and_unexecuted and not test_042_units_are_blank_before_launch',
], capture_output=True, text=True)
print(checks.stdout)
if checks.stderr:
    print(checks.stderr)
checks.check_returncode()
print('wiring checks passed')


....................................................................     [100%]
68 passed, 2 deselected in 28.56s

wiring checks passed


In [ ]:
subprocess.run([sys.executable, 'scripts/build_data.py'], check=True)


CompletedProcess(args=['/usr/bin/python3', 'scripts/build_data.py'], returncode=0)

## 4. Base and token audit

In [ ]:
import json, os, shutil, sys, time
sys.path.insert(0, os.getcwd())

from file_utils import read_json
from scripts.grpo_worker import stream_subprocess
from scripts.s42_worker import BASE_TAG, PILOT_TAG, config_path, pilot_passes

assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
CFG_PATH = config_path(0)
TECH = os.path.join(ROOT, 'results', '042_technical')
PILOT_DIR = os.path.join(TECH, 'pilot')
BASE_RESULT = os.path.join(ROOT, 'results', BASE_TAG)
BASE_DIR = os.path.join(ROOT, 'checkpoints', BASE_TAG)
BASE_RECORD = os.path.join(BASE_RESULT, '042_base_manifest.json')
LENGTH_RECORD = os.path.join(TECH, '042_token_audit.json')
os.makedirs(PILOT_DIR, exist_ok=True)
os.makedirs(BASE_RESULT, exist_ok=True)

def append_jsonl(path, row):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'a') as f:
        f.write(json.dumps(row) + '\n')

def run_worker(command, status_path, result_dir, label, test_passes=0):
    if os.path.exists(status_path):
        os.remove(status_path)
    started = time.time()
    process = stream_subprocess(command)
    status = read_json(status_path) if os.path.exists(status_path) else {
        'completed': False, 'error_type': 'MissingStatus',
        'error_message': 'worker exited without a status file'}
    status = {**status, 'label': label, 'returncode': process.returncode,
              'parent_wall_seconds': round(time.time() - started, 1)}
    append_jsonl(os.path.join(result_dir, '042_attempts.jsonl'), status)
    append_jsonl(os.path.join(RESULTS, 'compute_ledger.jsonl'), {
        'run_tag': label, 'condition': f'042_{status.get("task", "worker")}',
        'timings_sec': {'worker': status['parent_wall_seconds']},
        'train_sec': status['parent_wall_seconds'] if status.get('task') in ('pilot_sft', 'pilot', 'sft', 'grpo') else 0,
        'train_steps': status.get('completed_updates', 0),
        'mmlu_test_passes': test_passes,
        'completed': bool(status.get('completed')), 'gpu': status.get('gpu'),
    })
    return process, status

audit_status = os.path.join(TECH, 'audit_status.json')
if not os.path.exists(LENGTH_RECORD):
    process, status = run_worker([
        sys.executable, 'scripts/s42_worker.py', '--task', 'audit',
        '--config', CFG_PATH, '--status', audit_status,
        '--result-dir', TECH, '--record', LENGTH_RECORD,
    ], audit_status, TECH, '042_m124_token_audit')
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError(f'token audit failed: {status}')
print('completion cap:', read_json(LENGTH_RECORD)['completion_cap'])

convert_status = os.path.join(BASE_RESULT, 'convert_status.json')
if not os.path.exists(BASE_RECORD):
    local_base = '/content/042_m124_base_staging'
    if os.path.isdir(local_base):
        shutil.rmtree(local_base)
    process, status = run_worker([
        sys.executable, 'scripts/s42_worker.py', '--task', 'convert',
        '--config', CFG_PATH, '--status', convert_status,
        '--result-dir', BASE_RESULT, '--scratch-dir', local_base,
        '--out-dir', BASE_DIR, '--record', BASE_RECORD,
    ], convert_status, BASE_RESULT, '042_m124_base_conversion')
    if os.path.isdir(local_base):
        shutil.rmtree(local_base)
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError(f'base conversion failed: {status}')
print('base files:', len(read_json(BASE_RECORD)['artifact_sha256']))


completion cap: 512
config.json: 100% 665/665 [00:00<00:00, 2.86MB/s]

model.safetensors: downloading bytes:  43% 235M/548M [00:01<00:00, 338MB/s, 17.9MB/s  ]
model.safetensors: downloading bytes:  75% 412M/548M [00:01<00:00, 285MB/s, 36.5MB/s  ]
model.safetensors: downloading bytes:  87% 474M/548M [00:02<00:00, 160MB/s, 41.3MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:02<00:00, 177MB/s, 41.6MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:02<00:00, 205MB/s, 50.1MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 5716.78it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 522kB/s]
Writing model shards: 100% 1/1 [00:00<00:00,  3.85it/s]
base files: 7


## 5. Technical pilot

In [ ]:
REPORT_PATH = os.path.join(PILOT_DIR, 'pilot_report.json')
CLOSEOUT_PATH = os.path.join(PILOT_DIR, 'pilot_closeout.json')
PRECISION_PATH = os.path.join(PILOT_DIR, 'precision_report.json')
PILOT_SFT = '/content/042_m124_pilot_sft'
PILOT_GRPO = '/content/042_m124_pilot_grpo'

if not os.path.exists(REPORT_PATH):
    for path in (PILOT_SFT, PILOT_GRPO):
        if os.path.isdir(path):
            shutil.rmtree(path)
    sft_status_path = os.path.join(PILOT_DIR, 'pilot_sft_status.json')
    process, status = run_worker([
        sys.executable, 'scripts/s42_worker.py', '--task', 'pilot_sft',
        '--config', CFG_PATH, '--status', sft_status_path,
        '--result-dir', PILOT_DIR, '--base-dir', BASE_DIR, '--out-dir', PILOT_SFT,
    ], sft_status_path, PILOT_DIR, '042_m124_pilot_sft')
    if process.returncode != 0 or not status.get('completed'):
        raise RuntimeError(f'pilot SFT failed: {status}')

    pilot_status_path = os.path.join(PILOT_DIR, 'pilot_worker_status.json')
    process, status = run_worker([
        sys.executable, 'scripts/s42_worker.py', '--task', 'pilot',
        '--config', CFG_PATH, '--status', pilot_status_path,
        '--result-dir', PILOT_DIR, '--start-dir', PILOT_SFT,
        '--out-dir', PILOT_GRPO, '--precision-report', PRECISION_PATH,
        '--pilot-report', REPORT_PATH, '--pilot-closeout', CLOSEOUT_PATH,
    ], pilot_status_path, PILOT_DIR, '042_m124_pilot_grpo')
    if process.returncode != 0:
        raise RuntimeError(f'pilot failed: {status}')

report = read_json(REPORT_PATH)
closeout = read_json(CLOSEOUT_PATH)
assert pilot_passes(report, closeout), 'the experiment 042 pilot did not pass'
print('peak reserved GiB:', report['peak_reserved_gib'])
print('projected 500 updates:', round(report['projected_500_updates_hours'], 2), 'hours')
print('max completion:', report['precision_audit']['max_observed_completion_tokens'])
print('pilot passed')


Streaming output truncated to the last 5000 lines.
│ │               │ respective    │               │              │           │ │
│ │ According to  │ thought       │               │              │           │ │
│ │ Taylor, hard  │ (determinism… │               │              │           │ │
│ │ determinism   │ conformity),  │               │              │           │ │
│ │ conflicts     │ i.e to affirm │               │              │           │ │
│ │ with _____.   │ the injustice │               │              │           │ │
│ │ A. the        │ of decisions  │               │              │           │ │
│ │ findings of   │ to distribute │               │              │           │ │
│ │ science       │ preference    │               │              │           │ │
│ │ B. the fact   │ among         │               │              │           │ │
│ │ of            │ persons, i.e  │               │              │           │ │
│ │ deliberation  │ competition   │               │       

## 6. Close compute units

In [10]:
cu_after = 324.58
print('cu_after =', cu_after if cu_after is not None else 'FILL ME')
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif not os.path.exists(open_path):
    print('no open session record')
else:
    with open(open_path) as f:
        record = json.load(f)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['completed'] = True
    with open(UNITS_LOG, 'a') as f:
        f.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)


cu_after = 324.58
logged: {'session_id': '20260825_135245', 'ts_start': '2026-08-25 13:53', 'notebook': '042_pilot_colab_m124', 'colab_cu_before': 326.36, 'unit_rate': 5.3, 'ts_end': '2026-08-25 14:15', 'colab_cu_after': 324.58, 'colab_cu_used': 1.78, 'completed': True}
